# 01_stations : Stations piézométriques autour de Valence

### **Descriptif du Notebook VERSION VS CODE** :

**Le projet :**
- Connexion à l'API Hub'Eau sur le jeu de données "Piézométrie" (https://hubeau.eaufrance.fr/page/api-piezometrie)
- Données recherchées :
    - A. **ICI** : la liste des stations piézométriques de la zone → endpoint `/stations`
    - B. Notebook `02_chroniques` : le niveau d'eau des stations → endpoint `/chroniques`
- Zone géographique : le bassin de Valence (26000) = bbox « carré de 100 km centré sur Valence »

**Les étapes du Notebook :**

1. Imports
2. Configuration : variables du projet (.env) + constantes métier :
    - fichier .env = URL de l'API (un token y serait stocké si l'API en demandait un, ici pas nécessaire)
    - constantes métier = latitude/longitude de Valence, demi-côté du carré de la bbox
3. Calcul de la bbox
4. Fonction d'appel à l'API : définie dans `src/hubeau.py` (gestion des erreurs + pagination)
5. Récupération des stations piézométriques
6. Mise en DataFrame et contrôle qualité (shape, colonnes, valeurs manquantes)
7. Sauvegarde des données brutes (`data/raw/`)
8. Sauvegarde des stations (`data/processed/`) : stations avec mesures + totalité des stations
9. Export GeoJSON (cartographie : QGIS, geopandas) : totalité des stations + stations avec mesures

**Sorties :**
- `data/raw/stations_AAAA-MM-JJ.json` : données brutes de l'API
- `data/processed/stations_mesurees.csv` : stations avec mesures → **utilisé par `02_chroniques`**
- `data/processed/stations_total.csv` : les 176 stations, colonne `avec_mesures` (True / False)
- `data/processed/stations_total.geojson` : les 176 stations au format géographique, filtrables avec `avec_mesures`
- `data/processed/stations_mesurees.geojson` : uniquement les 60 stations avec mesures

### Prérequis avant d'utiliser le notebook

**Première installation (une seule fois)**
- Cloner le dépôt, puis dans le terminal, à la racine du projet :
  `python3 -m venv .venv` → `source .venv/bin/activate` → `pip install -r requirements.txt`
- Créer le fichier de variables : `cp .env.example .env`

**À chaque session de travail**
- Ouvrir le **dossier racine** `Hub-eau` dans VS Code (pas seulement le notebook)
- Récupérer les dernières modifications : `git pull`
- Vérifier que le **kernel** sélectionné (en haut à droite) est `.venv`
- Exécuter la cellule de vérification ci-dessous

**Pour installer une nouvelle bibliothèque**
- Activer le venv dans le terminal : `source .venv/bin/activate` (l'invite doit commencer par `(.venv)`)
- `pip install <bibliothèque>`, puis `pip freeze > requirements.txt`, puis commit

**Avant chaque commit**
- *Restart Kernel* + *Run All* : le notebook doit tourner en entier sans erreur
- `git status` : `.env`, `.venv/` et les fichiers de `data/` ne doivent jamais apparaître

### Réutilisation du notebook pour un autre territoire
- **Pour modifier les coordonnées GPS de la ville point central + dimension de la BBox**  
=> cellule "2. Configuration" > --- B. Constantes métier---  

- **Pour modifier les paramètres techniques de l'API**  
=> cellule "2. Configuration" > --- C. Paramètres techniques de l'API ---  

- **Pour modifier le format attendus de l'extraction (Json, Geojson, ou csv avec autre endpoint !)**  
=> Les données sont en Json mais la dernière cellule exporte en GeoJson

In [1]:
# Vérification de l'environnement : le kernel doit être celui du .venv du projet
import sys

print("Python utilisé :", sys.executable)
assert ".venv" in sys.executable, "⚠️ Mauvais kernel : sélectionner le .venv en haut à droite"

Python utilisé : /home/data-08/Documents/00-PROJETS/Hub-eau/.venv/bin/python


## **Appel API - phase A /stations**

In [2]:
# 1. Imports des bibliothèques

# Rechargement automatique des modules de src/ quand ils sont modifiés
%load_ext autoreload
%autoreload 2

# Bibliothèque standard (livrée avec Python)
import json                         # sauvegarde des données brutes
import os                           # lecture des variables d'environnement
import sys                          # chemin de recherche des modules
from datetime import datetime       # horodatage des fichiers sauvegardés
from pathlib import Path            # chemins de fichiers (data/raw/...)

# Bibliothèques externes (installées dans le .venv, listées dans requirements.txt)
import geopandas as gpd            # données géographiques (export GeoJSON)
import pandas as pd                 # mise en DataFrame et contrôle des données
from dotenv import load_dotenv      # chargement du fichier .env

# Modules locaux (dossier src/ à la racine du projet)
sys.path.append(str(Path.cwd().parent))     # rend la racine du projet visible pour Python
from src.hubeau import appeler_api, bbox_autour

In [3]:
# 2. Configuration

# --- A. Variables d'environnement (fichier .env) ---
load_dotenv()                                   # charge le .env (cherché en remontant les dossiers)

BASE_URL = os.getenv("HUBEAU_BASE_URL")
if BASE_URL is None:
    raise ValueError("HUBEAU_BASE_URL introuvable : vérifier que le fichier .env existe à la racine du projet")

URL_STATIONS = f"{BASE_URL}/stations"           # liste des stations piézométriques

# --- B. Constantes métier ---
VALENCE_LAT = 44.9334                           # latitude du centre de la zone (WGS84)
VALENCE_LON = 4.8924                            # longitude du centre de la zone (WGS84)
DEMI_COTE_KM = 50                               # demi-côté du carré => carré de 100 km

# --- C. Paramètres techniques de l'API ---
PAGE_SIZE = 1000                                # nombre de résultats par page
TIMEOUT_S = 30                                  # délai max d'attente d'une réponse (secondes)
PAUSE_S = 0.2                                   # pause entre deux pages (respect de l'API)

# --- D. Chemins du projet ---
PROJECT_ROOT = Path.cwd().parent                # le notebook est dans notebooks/ => on remonte d'un niveau
assert (PROJECT_ROOT / "requirements.txt").exists(), f"Racine du projet incorrecte : {PROJECT_ROOT}"

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW.mkdir(parents=True, exist_ok=True)     # créé s'il n'existe pas
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# --- Récapitulatif ---
print("URL stations     :", URL_STATIONS)
print("Centre           :", VALENCE_LAT, VALENCE_LON, f"(demi-côté {DEMI_COTE_KM} km)")
print("Racine projet    :", PROJECT_ROOT)
print("Données brutes   :", DATA_RAW)
print("Données traitées :", DATA_PROCESSED)

URL stations     : https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations
Centre           : 44.9334 4.8924 (demi-côté 50 km)
Racine projet    : /home/data-08/Documents/00-PROJETS/Hub-eau
Données brutes   : /home/data-08/Documents/00-PROJETS/Hub-eau/data/raw
Données traitées : /home/data-08/Documents/00-PROJETS/Hub-eau/data/processed


#### 3. Zone d'étude : bbox autour de Valence

Carré de **100 km de côté** (demi-côté `DEMI_COTE_KM` = 50 km) centré sur Valence,
calculé avec `bbox_autour()` (voir `src/hubeau.py`).

In [4]:
# 3. Calcul de la bbox (fonction : src/hubeau.py)

BBOX = bbox_autour(VALENCE_LAT, VALENCE_LON, DEMI_COTE_KM)
print("BBox :", BBOX)

# Contrôle visuel de la zone sur une carte
min_lon, min_lat, max_lon, max_lat = BBOX.split(",")
print(f"Vérifier : https://bboxfinder.com/#{min_lat},{min_lon},{max_lat},{max_lon}")

BBox : 4.2579,44.4842,5.5269,45.3826
Vérifier : https://bboxfinder.com/#44.4842,4.2579,45.3826,5.5269


#### 4. Fonction d'appel à l'API

La fonction `appeler_api()` est définie dans **`src/hubeau.py`** (importée en cellule 1).

Elle gère :
- la **pagination** : suit le lien `next` renvoyé par l'API jusqu'à la dernière page ;
- les **erreurs** : accepte les codes 200 (complet) et 206 (partiel), lève une erreur sinon ;
- un **délai d'attente** (`timeout_s`) et une **pause** entre deux pages (`pause_s`) ;
- une **sécurité** `max_pages` pour éviter un nombre d'appels excessif.

⚠️ Limite Hub'Eau : `page × size` ne peut pas dépasser 20 000 résultats par requête.


In [5]:
# 5. Récupération des stations piézométriques

params_stations = {
    "bbox": BBOX,           # zone géographique calculée à l'étape 3
    "size": PAGE_SIZE,      # nombre de résultats par page (configuration)
    "format": "json",
}
# NB : la documentation liste d'autres filtres possibles pour /stations, comme code_departement ou date_recherche.

stations = appeler_api(URL_STATIONS, params_stations,
                       timeout_s=TIMEOUT_S, pause_s=PAUSE_S)

print(f"\n{len(stations)} stations récupérées")

Page 1 : 176 résultats (176/176)

176 stations récupérées


In [6]:
# Visualiser le résultat :
stations[0]

{'code_bss': '07702X0081/1',
 'urn_bss': 'http://services.ades.eaufrance.fr/pointeau/07702X0081/1',
 'date_debut_mesure': None,
 'date_fin_mesure': None,
 'code_commune_insee': '26325',
 'nom_commune': "Saint-Rambert-d'Albon",
 'x': 4.83728378,
 'y': 45.289165654,
 'codes_bdlisa': None,
 'urns_bdlisa': [],
 'geometry': {'type': 'Point',
  'crs': {'type': 'name',
   'properties': {'name': 'urn:ogc:def:crs:OGC:1.3:CRS84'}},
  'coordinates': [4.83728377989336, 45.2891656535815]},
 'bss_id': 'BSS001WLSH',
 'altitude_station': '164.0',
 'nb_mesures_piezo': 0,
 'code_departement': '26',
 'nom_departement': 'Drôme',
 'libelle_pe': None,
 'profondeur_investigation': 27.25,
 'codes_masse_eau_edl': None,
 'noms_masse_eau_edl': None,
 'urns_masse_eau_edl': [],
 'date_maj': 'Wed Jun 25 20:01:28 CEST 2025'}

#### 6. Mise en DataFrame et contrôle qualité
   6a. Création du DataFrame
   6b. Contrôle qualité : types, valeurs manquantes, doublons
   6c. Conversion des types
   6d. Sélection des stations avec mesures

In [7]:
# 6a. Création du DataFrame

df_stations = pd.DataFrame(stations)

print("Dimensions (lignes, colonnes) :", df_stations.shape)
df_stations.head(3)

Dimensions (lignes, colonnes) : (176, 22)


,code_bss,urn_bss,date_debut_mesure,date_fin_mesure,code_commune_insee,nom_commune,x,y,codes_bdlisa,urns_bdlisa,...,altitude_station,nb_mesures_piezo,code_departement,nom_departement,libelle_pe,profondeur_investigation,codes_masse_eau_edl,noms_masse_eau_edl,urns_masse_eau_edl,date_maj
0,07702X0081/1,http://services.ades.eaufrance.fr/pointeau/077...,NaN,NaN,26325,Saint-Rambert-d'Albon,4.837284,45.289166,None,[],...,164.0,0,26,Drôme,NaN,27.25,None,None,[],Wed Jun 25 20:01:28 CEST 2025
1,07477X0032/3821,http://services.ades.eaufrance.fr/pointeau/074...,NaN,NaN,38130,La Côte-Saint-André,5.274229,45.373098,None,[],...,349.0,0,38,Isère,NaN,20.50,None,None,[],Wed Jun 25 20:01:28 CEST 2025
2,08422X0190/F1,http://services.ades.eaufrance.fr/pointeau/084...,1999-12-17,2026-11-13,26166,Loriol-sur-Drôme,4.791298,44.756364,"[529AA00, 712BM01]",[http://reseau.eaufrance.fr/geotraitements/bdl...,...,95.0,9157,26,Drôme,PIEZOMETRE - AUTOROUTE (LORIOL SUR DROME - BRG...,11.50,[DG337],[Alluvions de la Drôme],[http://www.sandre.eaufrance.fr/geo/MasseDEauS...,Sun Jul 14 13:00:02 CEST 2024


In [8]:
# 6b. Contrôle qualité : types, valeurs manquantes, doublons

# Valeurs manquantes par colonne (uniquement celles qui en ont)
manquants = df_stations.isna().sum()
print("\nValeurs manquantes :\n", manquants[manquants > 0].sort_values(ascending=False))
print("\n")
      
# Types des colonnes
df_stations.info()

# Doublons sur l'identifiant de station
print("\nDoublons sur code_bss :", df_stations["code_bss"].duplicated().sum())


Valeurs manquantes :
 date_debut_mesure           116
date_fin_mesure             116
noms_masse_eau_edl          111
codes_masse_eau_edl         111
codes_bdlisa                 91
libelle_pe                   41
profondeur_investigation     31
dtype: int64


<class 'pandas.DataFrame'>
RangeIndex: 176 entries, 0 to 175
Data columns (total 22 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   code_bss                  176 non-null    str    
 1   urn_bss                   176 non-null    str    
 2   date_debut_mesure         60 non-null     str    
 3   date_fin_mesure           60 non-null     str    
 4   code_commune_insee        176 non-null    str    
 5   nom_commune               176 non-null    str    
 6   x                         176 non-null    float64
 7   y                         176 non-null    float64
 8   codes_bdlisa              85 non-null     object 
 9   urns_bdlisa               176 non-

**Constat :** 116 stations sur 176 ont `nb_mesures_piezo = 0` (valeur renseignée mais égale à zéro).
Elles n'ont donc pas de dates de mesure, ce qui explique les 116 valeurs manquantes sur
`date_debut_mesure` et `date_fin_mesure`. Aucun doublon sur `code_bss`.

In [9]:
# 6c. Conversion des types

df_stations["altitude_station"] = pd.to_numeric(df_stations["altitude_station"], errors="coerce")
df_stations["date_debut_mesure"] = pd.to_datetime(df_stations["date_debut_mesure"])
df_stations["date_fin_mesure"] = pd.to_datetime(df_stations["date_fin_mesure"])

df_stations[["altitude_station", "date_debut_mesure", "date_fin_mesure"]].dtypes


altitude_station            float64
date_debut_mesure    datetime64[us]
date_fin_mesure      datetime64[us]
dtype: object

In [10]:
# 6d. Sélection des stations ayant au moins une mesure de niveau d'eau

df_stations_mesurees = df_stations[df_stations["nb_mesures_piezo"] > 0].copy()

print(f"{len(df_stations_mesurees)} stations avec mesures sur {len(df_stations)}")
df_stations_mesurees[["code_bss", "nom_commune", "nom_departement",
                      "date_debut_mesure", "date_fin_mesure", "nb_mesures_piezo"]].head(10)


60 stations avec mesures sur 176


,code_bss,nom_commune,nom_departement,date_debut_mesure,date_fin_mesure,nb_mesures_piezo
2,08422X0190/F1,Loriol-sur-Drôme,Drôme,1999-12-17,2026-11-13,9157
4,07948X0047/F1,Romans-sur-Isère,Drôme,2009-06-12,2026-11-13,6136
5,08664X0069/F1,Soyans,Drôme,2008-12-03,2026-11-11,6526
6,07703X0064/P1,Anneyron,Drôme,2009-06-09,2026-04-20,6155
8,08187X0238/P1,Beaumont-lès-Valence,Drôme,2009-06-26,2026-11-13,6343
9,07955X0099/P1,Chatuzange-le-Goubet,Drôme,2005-11-24,2026-11-13,7450
12,08184X0084/PZ1,Valence,Drôme,1982-02-04,2026-10-21,9637
14,08422X0191/F2,Livron-sur-Drôme,Drôme,1999-12-17,2026-11-12,8469
17,08424X0006/F2,Eurre,Drôme,1995-06-29,2026-11-13,8900
20,08426X0133/P2,Saulce-sur-Rhône,Drôme,2009-06-17,2026-11-13,6249


In [11]:
# 7. Sauvegarde des données brutes (data/raw/)

date_extraction = datetime.now()
fichier_brut = DATA_RAW / f"stations_{date_extraction:%Y-%m-%d}.json"

# On enregistre les données ET le contexte de l'extraction (traçabilité)
contenu_brut = {
    "date_extraction": date_extraction.isoformat(timespec="seconds"),
    "url": URL_STATIONS,
    "params": params_stations,
    "nb_resultats": len(stations),
    "data": stations,
}

with open(fichier_brut, "w", encoding="utf-8") as f:
    json.dump(contenu_brut, f, ensure_ascii=False, indent=2)

print(f"✅ {len(stations)} stations sauvegardées dans : {fichier_brut}")


✅ 176 stations sauvegardées dans : /home/data-08/Documents/00-PROJETS/Hub-eau/data/raw/stations_2026-11-16.json


In [12]:
# Vérification : relecture du fichier sauvegardé
with open(fichier_brut, encoding="utf-8") as f:
    relu = json.load(f)

print("Date d'extraction :", relu["date_extraction"])
print("Nombre de stations :", len(relu["data"]))
assert len(relu["data"]) == len(stations), "⚠️ Le fichier relu ne correspond pas aux données récupérées"

Date d'extraction : 2026-11-16T11:04:15
Nombre de stations : 176


In [13]:
# 8. Sauvegarde des données traitées (data/processed/) : entrée du notebook 02_chroniques

fichier_stations = DATA_PROCESSED / "stations_mesurees.csv"
df_stations_mesurees.to_csv(fichier_stations, index=False)

print(f"✅ {len(df_stations_mesurees)} stations avec mesures sauvegardées dans : {fichier_stations}")

✅ 60 stations avec mesures sauvegardées dans : /home/data-08/Documents/00-PROJETS/Hub-eau/data/processed/stations_mesurees.csv


In [14]:
# 8b. Sauvegarde de la totalité des stations, avec un indicateur de présence de mesures

df_stations_total = df_stations.copy()
df_stations_total["avec_mesures"] = df_stations_total["nb_mesures_piezo"] > 0   # True si au moins 1 mesure

fichier_stations_total = DATA_PROCESSED / "stations_total.csv"
df_stations_total.to_csv(fichier_stations_total, index=False)

print(f"✅ {len(df_stations_total)} stations sauvegardées dans : {fichier_stations_total}")
print(df_stations_total["avec_mesures"].value_counts())

✅ 176 stations sauvegardées dans : /home/data-08/Documents/00-PROJETS/Hub-eau/data/processed/stations_total.csv
avec_mesures
False    116
True      60
Name: count, dtype: int64


#### 9. Export GeoJSON

Conversion des stations en **GeoDataFrame** (geopandas) puis export au format **GeoJSON**,
lisible directement par les outils cartographiques (QGIS, geopandas, folium, geojson.io…).

**2 fichiers GeoJSON sont produits :**

| Fichier | Contenu | Usage |
|---|---|---|
| `stations_total.geojson` | Les **176 stations**, avec la colonne `avec_mesures` (True / False) | Carte de toutes les stations, avec un **filtre** ou une couleur selon `avec_mesures` |
| `stations_mesurees.geojson` | Uniquement les **60 stations avec mesures** | Carte des seules stations exploitables pour les chroniques |

- Le second fichier est obtenu en **filtrant le premier** sur `avec_mesures` : les deux restent cohérents.
- La géométrie (un point par station) est construite à partir des colonnes `x` (longitude) et `y` (latitude).
- `crs="EPSG:4326"` : système de coordonnées WGS84 (degrés), celui utilisé par l'API Hub'Eau.
- Seules les colonnes simples sont exportées : les colonnes contenant des listes ou des dictionnaires
  (`geometry` d'origine, `urns_*`, `codes_*`…) restent disponibles dans le fichier brut JSON.

In [15]:
# 9. Export GeoJSON (data/processed/) : totalité des stations + stations avec mesures

COLONNES_GEO = [
    "code_bss", "bss_id", "nom_commune", "code_commune_insee",
    "code_departement", "nom_departement",
    "altitude_station", "profondeur_investigation", "nb_mesures_piezo",
    "date_debut_mesure", "date_fin_mesure",
    "x", "y",
    "avec_mesures",                     # True / False : filtre pour la cartographie
]

# 9a. Toutes les stations (176), avec la colonne avec_mesures
gdf_stations_total = gpd.GeoDataFrame(
    df_stations_total[COLONNES_GEO],
    geometry=gpd.points_from_xy(df_stations_total["x"], df_stations_total["y"]),
    crs="EPSG:4326",                    # WGS84 : longitude / latitude en degrés
)

# 9b. Uniquement les stations avec mesures (60) : filtre du fichier total
gdf_stations_mesurees = gdf_stations_total[gdf_stations_total["avec_mesures"]].copy()

# Export des 2 fichiers
fichier_geojson_total = DATA_PROCESSED / "stations_total.geojson"
fichier_geojson_mesurees = DATA_PROCESSED / "stations_mesurees.geojson"

gdf_stations_total.to_file(fichier_geojson_total, driver="GeoJSON")
gdf_stations_mesurees.to_file(fichier_geojson_mesurees, driver="GeoJSON")

print(f"✅ {len(gdf_stations_total)} stations (toutes) exportées dans    : {fichier_geojson_total}")
print(f"✅ {len(gdf_stations_mesurees)} stations (avec mesures) exportées dans : {fichier_geojson_mesurees}")

✅ 176 stations (toutes) exportées dans    : /home/data-08/Documents/00-PROJETS/Hub-eau/data/processed/stations_total.geojson
✅ 60 stations (avec mesures) exportées dans : /home/data-08/Documents/00-PROJETS/Hub-eau/data/processed/stations_mesurees.geojson


In [16]:
# Vérification : relecture des 2 fichiers GeoJSON

for fichier, gdf_attendu in [(fichier_geojson_total, gdf_stations_total),
                             (fichier_geojson_mesurees, gdf_stations_mesurees)]:
    gdf_relu = gpd.read_file(fichier)
    print(f"--- {fichier.name}")
    print("  Nombre de stations     :", len(gdf_relu))
    print("  Système de coordonnées :", gdf_relu.crs)
    print("  Type de géométrie      :", gdf_relu.geom_type.unique().tolist())
    print("  avec_mesures           :", gdf_relu["avec_mesures"].value_counts().to_dict())
    assert len(gdf_relu) == len(gdf_attendu), f"⚠️ {fichier.name} ne correspond pas aux données exportées"

--- stations_total.geojson
  Nombre de stations     : 176
  Système de coordonnées : EPSG:4326
  Type de géométrie      : ['Point']
  avec_mesures           : {False: 116, True: 60}
--- stations_mesurees.geojson
  Nombre de stations     : 60
  Système de coordonnées : EPSG:4326
  Type de géométrie      : ['Point']
  avec_mesures           : {True: 60}
